# 01. Spatiotemporal tensor geometry

![Video tensors become tubelet tokens](../images/01_spatiotemporal_tensor_geometry.svg)

A video tensor is an address book. Every number in it is found by answering five
questions: which clip, which color channel, which moment, which row, which column. This
notebook makes that idea executable.

**What you will do:** read `(B,C,T,H,W)` shapes, compute `Conv3d` output sizes, turn a
token grid into a sequence, map a flat index back to coordinates, and use gather and
scatter safely. Everything is synthetic and runs on CPU in a few seconds.

Read the [lecture](../lectures/01_spatiotemporal_tensor_geometry.md) first if any step
below feels unmotivated.

In [ ]:
import random
import numpy as np
import torch

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cpu')
print(f'torch={torch.__version__}, device={device}')

## 1. Name every axis

Start with the address book itself. A video batch has shape
`(batch, channel, time, height, width)`, and every index you supply answers one of the
five questions.

Three operations behave differently, and mixing them up causes most shape bugs:

- an integer index answers a question and **removes** that axis,
- a slice leaves the question partly open and **keeps** the axis,
- `permute` **reorders** axes, while `reshape` only regroups the underlying run of
  values in memory.

The cell below asserts each of those claims.

In [ ]:
B, C, T, H, W = 2, 3, 8, 16, 16
video = torch.arange(B*C*T*H*W, dtype=torch.float32).reshape(B, C, T, H, W)
frame = video[0, :, 2]                 # integer indices remove B and T
one_frame_batch = video[0:1, :, 2:3]   # slices retain singleton axes
assert frame.shape == (C, H, W)
assert one_frame_batch.shape == (1, C, 1, H, W)
channels_last = video.permute(0, 2, 3, 4, 1)
assert channels_last.shape == (B, T, H, W, C)
print('video:', tuple(video.shape), 'frame:', tuple(frame.shape))

## 2. Compute a tubelet grid

Now that the axes have names, we can summarize local blocks of video. A **tubelet** is a
small box covering a few frames and a patch of each frame, and one convolution filter
turns each tubelet into a single learned feature vector called a token.

The count of tubelets along one axis is the count of valid kernel placements. For input
length $L$, kernel $K$, stride $S$, padding $P$, and dilation $A$, that count is
$\lfloor(L+2P-A(K-1)-1)/S+1\rfloor$. Setting the kernel equal to the stride makes the
tubelets non-overlapping, which keeps the arithmetic easy to check by hand.

Keep two scales apart while reading the code. A **training window** is the whole clip
selected from a longer source sequence. **Tubelets** are the small blocks inside that
clip. Window overlap changes data support; tubelet overlap changes encoder computation.

In [ ]:
def conv_output(length, kernel, stride, padding=0, dilation=1):
    return (length + 2*padding - dilation*(kernel - 1) - 1) // stride + 1

D = 12
kernel = stride = (2, 4, 4)
embed = torch.nn.Conv3d(C, D, kernel_size=kernel, stride=stride, bias=False)
x = torch.randn(B, C, T, H, W, device=device)
grid = embed(x)
expected_grid = (B, D, conv_output(T, 2, 2), conv_output(H, 4, 4), conv_output(W, 4, 4))
assert tuple(grid.shape) == expected_grid == (2, 12, 4, 4, 4)
print('tubelet grid:', tuple(grid.shape))

## 3. Grid to sequence and back

The convolution hands back a grid, but attention layers expect a flat list of tokens.
The conversion is pure bookkeeping, so nothing about the values changes.

`flatten(2)` merges the three grid axes into one while keeping the batch and feature
axes. `transpose(1, 2)` then moves features last to reach the usual `(B, N, D)`
convention. Both steps typically return views rather than copies, and reversing them
must recover the original grid exactly. The assertion below is the test of that claim.

In [ ]:
Bt, Dt, Tt, Ht, Wt = grid.shape
tokens = grid.flatten(2).transpose(1, 2)
N = Tt * Ht * Wt
assert tokens.shape == (Bt, N, Dt)
restored = tokens.transpose(1, 2).reshape(Bt, Dt, Tt, Ht, Wt)
torch.testing.assert_close(restored, grid)
print('sequence:', tuple(tokens.shape), 'round trip exact:', torch.equal(restored, grid))

## 4. Flat token indices preserve coordinates

Flattening only stays harmless if we can still say where each token came from. With
row-major layout the width coordinate changes fastest, then height, then time, so the
flat index is `n = (t*H_grid + h)*W_grid + w`.

Notice that this is the stride rule again: the coefficients `H_grid*W_grid`, `W_grid`,
and `1` are strides in token units. Integer division and remainders peel the address
back apart, one scale at a time. The loop below checks the round trip for every token.

In [ ]:
def flatten_coord(t, h, w, grid_h, grid_w):
    return (t * grid_h + h) * grid_w + w

def unflatten_index(n, grid_h, grid_w):
    t, remainder = divmod(n, grid_h * grid_w)
    h, w = divmod(remainder, grid_w)
    return t, h, w

for n in range(N):
    coord = unflatten_index(n, Ht, Wt)
    assert flatten_coord(*coord, Ht, Wt) == n
print('token 27 ->', unflatten_index(27, Ht, Wt))

## 5. Gather and scatter complete feature vectors

Masked prediction needs a subset of tokens, so the last skill is selecting tokens by
index and putting them back.

`torch.gather` requires an index tensor with the same rank as the input, so a `(B, M)`
index of token positions must first grow a feature axis. `unsqueeze` adds that axis with
length one, and `expand` broadcasts it to width `D` without allocating repeated index
storage. `scatter_` writes values back when the indices are unique. For repeated indices
it simply overwrites, so use `scatter_add_` or an explicit reduction instead.

In [ ]:
indices = torch.tensor([[0, 5, 27], [2, 11, 63]])
expanded = indices.unsqueeze(-1).expand(-1, -1, Dt)
selected = torch.gather(tokens, dim=1, index=expanded)
assert selected.shape == (B, 3, D)
canvas = torch.zeros_like(tokens)
canvas.scatter_(dim=1, index=expanded, src=selected)
for b in range(B):
    torch.testing.assert_close(canvas[b, indices[b]], tokens[b, indices[b]])
assert torch.count_nonzero(canvas).item() <= B * 3 * D
print('selected:', tuple(selected.shape), 'nonzero canvas entries:', torch.count_nonzero(canvas).item())

## Efficiency, exercises, and takeaways

**Efficiency.** `Conv3d` is far faster than a Python loop over tubelets. Prefer `expand`
to `repeat`, since only `repeat` allocates. Call `contiguous()` only when an API demands
contiguous storage, and assert shapes at every layout boundary.

**Exercises:** (1) Change the input to `(2,3,10,20,20)` and the kernel and stride to
`(2,5,5)`, and predict the grid before you run it. (2) Select five indices per sample and
verify a gather and scatter round trip. (3) Create repeated indices and compare the
overwrite behavior of `scatter_` with `scatter_add_`.

**Takeaways:** shape is meaning, not bookkeeping; tubelets produce a learned local token
grid; flattening preserves a coordinate convention you must record; gather and scatter
move batched tokens efficiently once the index shapes line up.

## Continue learning

[Lecture](../lectures/01_spatiotemporal_tensor_geometry.md) | [Curriculum](../README.md) | [Next notebook: 02](02_inner_product_geometry.ipynb)